# Tidyverse 데이터 가공 및 정제

언어 간 교차 데이터 파이프라인: Python으로 데이터 다운로드 → R dplyr로 데이터 처리 → Python으로 결과 시각화.

Python과 R이 파일을 교환할 수 있게 해주는 공유 파일 시스템인 **SharedVFS**를 시연합니다.

## 1. Python: 데이터셋 다운로드

In [ ]:
import micropip
await micropip.install('pandas')
import pandas as pd, pyodide.http, os

url = "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv"
resp = await pyodide.http.pyfetch(url)
text = await resp.string()

os.makedirs("/shared/data", exist_ok=True)
with open("/shared/data/gapminder.csv", "w") as f:
    f.write(text)

df = pd.read_csv("/shared/data/gapminder.csv")
print(f"Downloaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. R: dplyr + tidyr 설치 및 공유 데이터 읽기

In [ ]:
install.packages(c("dplyr", "tidyr"))
library(dplyr)

gap <- read.csv("/shared/data/gapminder.csv")
cat("Read from SharedVFS:", nrow(gap), "rows\n")
glimpse(gap)

## 3. dplyr: 대륙별 요약 (2007년)

In [ ]:
gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    countries = n(),
    mean_life = round(mean(lifeExp), 1),
    median_gdp = round(median(gdpPercap), 0),
    total_pop = sum(as.numeric(pop))
  ) %>%
  arrange(desc(mean_life))

## 4. dplyr: 기대 수명 증가폭 상위 국가

In [ ]:
gains <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  select(country, continent, year, lifeExp) %>%
  tidyr::pivot_wider(names_from = year, values_from = lifeExp,
                     names_prefix = "y") %>%
  mutate(gain = y2007 - y1952) %>%
  arrange(desc(gain)) %>%
  head(10)
gains

## 5. dplyr: 대륙별 인구 성장

In [ ]:
pop_growth <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  group_by(continent, year) %>%
  summarize(total_pop = sum(as.numeric(pop)), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = year, values_from = total_pop,
                     names_prefix = "pop_") %>%
  mutate(growth_pct = round((pop_2007 / pop_1952 - 1) * 100, 1)) %>%
  arrange(desc(growth_pct))
pop_growth

## 6. R: 결과를 SharedVFS에 쓰기

In [ ]:
# Write continent summary for Python to visualize
summary_2007 <- gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    mean_life = round(mean(lifeExp), 1),
    mean_gdp = round(mean(gdpPercap), 0),
    .groups = "drop"
  )
write.csv(summary_2007, "/shared/data/r_summary.csv", row.names = FALSE)
cat("Wrote /shared/data/r_summary.csv\n")
summary_2007

## 7. Python: R의 처리 결과 시각화

In [ ]:
import micropip
await micropip.install('plotly')
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

r_summary = pd.read_csv("/shared/data/r_summary.csv")
print("Read from SharedVFS (written by R):")
print(r_summary.to_string(index=False))

fig = px.bar(r_summary, x="continent", y="mean_life",
             title="Mean Life Expectancy by Continent (2007) — from R → Python",
             labels={"mean_life": "Life Expectancy (years)", "continent": "Continent"},
             color="continent")
fig.update_layout(template='plotly_dark', showlegend=False)
show_plotly(fig)

In [ ]:
fig = px.scatter(r_summary, x="mean_gdp", y="mean_life",
                 text="continent", size=[40]*len(r_summary),
                 title="GDP vs Life Expectancy by Continent (R summary → Python plot)",
                 labels={"mean_gdp": "Mean GDP per Capita", "mean_life": "Mean Life Expectancy"})
fig.update_traces(textposition="top center")
fig.update_layout(template='plotly_dark')
fig.update_yaxes(range=[r_summary['mean_life'].min() - 2, r_summary['mean_life'].max() + 6])
show_plotly(fig)

## 핵심 요점

- **Python**이 CSV 데이터를 `/shared/data/`에 다운로드함
- **R**이 SharedVFS를 통해 데이터를 읽고 dplyr 파이프라인으로 처리함
- **R**이 요약 결과를 `/shared/data/r_summary.csv`에 다시 저장함
- **Python**이 R의 출력을 읽어 인터랙티브 Plotly 차트를 생성함

모든 파일 공유는 SharedVFS를 통해 이루어집니다 — 수동 가져오기나 내보내기가 필요 없습니다.